# 14b — Engineer Derived Features

Sits between **Notebook 14** (feature matrix assembly) and **Notebook 15** (LASSO feature
selection). Reads the country-year feature matrix from ADLS, adds derived columns, and
writes an augmented parquet that Notebooks 15 and 16 consume in place of the raw matrix.

Run this notebook after 14 and before 15. Notebook 15's `FEATURE_MATRIX_PREFIX` should
point to the `feature_matrix_engineered` path written here.

## Sections

| Section | What it adds |
|---|---|
| **A — Transformations** | Log, square-root, polynomial expansions, first differences, HP-filter trend/cycle |
| **B — Interactions** | Configured pairwise products capturing cross-domain effects |
| **C — Spatial spillover** | Neighbor-weighted spatial lags, local Moran's I, LISA cluster indicators |

Each section is toggled independently in `ENG_CFG`. A feature catalog CSV is written
alongside the output parquet documenting every column added, its source variables, and
the transformation applied.

### Spatial data requirement

Section C requires a CSV at `ENG_CFG['geo_coords_path']` with columns `iso3, lat, lon`
(decimal degrees, one row per country). A suitable file can be built from Natural Earth
populated places or the `world_cities` dataset in GeoPandas. The section is skipped
gracefully if the file is absent.

In [ ]:
%%capture
%pip install \
    'azure-identity>=1.15.0' \
    'adlfs>=2023.9.0' \
    'pandas>=2.0' \
    numpy \
    scipy \
    'statsmodels>=0.14.0' \
    'libpysal>=4.6.0' \
    'esda>=2.4.0' \
    'ruptures>=1.1.0' \
    pyarrow --quiet

In [ ]:
import os
import warnings
import logging
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats

import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')
logger = logging.getLogger(__name__)

RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ENG_CFG = {
    # ── ADLS ─────────────────────────────────────────────────────────────────
    'adls_account_name': os.environ['ADLS_ACCOUNT_NAME'],
    'adls_container':    os.getenv('ADLS_CONTAINER', 'data'),

    # Input: latest date-partitioned parquet from notebook 14
    # Set to None to auto-detect the latest partition.
    'feature_matrix_prefix': 'processed/feature_matrix',
    'labels_prefix':         'processed/feature_matrix',
    'feature_matrix_date':   None,   # e.g. '20240315'; None = auto-latest

    # Spatial coordinates: CSV with columns iso3, lat, lon
    # Provide a path relative to this notebook or an absolute path.
    # Set to None to skip Section C entirely.
    'geo_coords_path': '../data/geo_coords.csv',

    # Output prefix on ADLS
    'output_prefix': 'processed/feature_matrix_engineered',

    # ── Section toggles ───────────────────────────────────────────────────────
    'enable_transformations': True,
    'enable_interactions':    True,
    'enable_spatial':         True,

    # ── A: Transformations ────────────────────────────────────────────────────
    # Columns to log-transform (right-skewed counts and levels).
    # log1p is used so zero values are handled safely.
    'log_cols': [
        'gdp_per_capita', 'population', 'gdp_total',
        'military_expenditure_usd', 'acled_fatalities_annual',
        'acled_events_battles_annual', 'unhcr_refugees_hosted',
        'unhcr_idps',
    ],

    # Columns for which to add a square-root transform (bounded counts).
    'sqrt_cols': [
        'acled_events_protests_annual', 'acled_events_riots_annual',
        'nelda_elections_count',
    ],

    # First-difference columns: captures year-on-year change
    # (computed within-country, lag-1 subtraction).
    'diff_cols': [
        'gdp_per_capita', 'polity_score', 'v2x_libdem',
        'fsi_total', 'inflation_cpi', 'sipri_milex_gdp_pct',
    ],

    # HP-filter decomposition: adds _trend and _cycle for each listed column.
    # Uses lambda=6.25 (standard for annual data).
    'hp_cols': [
        'gdp_per_capita', 'inflation_cpi', 'sipri_milex_gdp_pct', 'fsi_total',
    ],
    'hp_lambda': 6.25,

    # ── B: Interaction effects ────────────────────────────────────────────────
    # Each entry: (col_a, col_b, output_name).
    # The new column is col_a * col_b (both standardised first so the product
    # is interpretable in standard-deviation units).
    'interaction_pairs': [
        # Economic stress × political exclusion
        ('gdp_per_capita',          'v2x_libdem',             'gdp_x_libdem'),
        ('inflation_cpi',           'polity_score',           'inflation_x_polity'),
        # Food insecurity × prior conflict
        ('fao_food_price_index',    'hist_conflict_years',    'food_x_conflict_history'),
        # Military capacity × political instability
        ('sipri_milex_gdp_pct',     'fsi_security_apparatus', 'milex_x_fsi_security'),
        # Ethnic fractionalization × political exclusion
        ('ethnic_fractionalization', 'v2x_liberal',           'ethnic_x_liberal'),
        # Refugee pressure × state fragility
        ('unhcr_refugees_hosted',   'fsi_total',              'refugees_x_fragility'),
        # Leadership tenure × irregular exit history
        ('archigos_tenure_years',   'archigos_irreg_entry',   'tenure_x_irreg_entry'),
    ],

    # ── C: Spatial ────────────────────────────────────────────────────────────
    'spatial_k_neighbors': 5,

    # Features for which to compute a spatial lag (neighbor-weighted mean)
    # and local Moran's I / LISA cluster type.
    'spatial_features': [
        'gdp_per_capita', 'polity_score', 'fsi_total',
        'acled_fatalities_annual', 'unhcr_refugees_hosted',
        'sipri_milex_gdp_pct',
    ],

    # ── Schema ────────────────────────────────────────────────────────────────
    'country_col': 'iso3',
    'year_col':    'year',
    'outcome_cols': [
        'civil_war_onset', 'coup_attempt', 'regime_backsliding',
        'mass_unrest_onset', 'humanitarian_crisis_onset',
    ],
}

print('ENG_CFG loaded.')
print(f"Output prefix : {ENG_CFG['output_prefix']}/{RUN_DATE}/")

In [ ]:
# ADLS helpers
credential = DefaultAzureCredential()
fs = adlfs.AzureBlobFileSystem(
    account_name=ENG_CFG['adls_account_name'],
    credential=credential,
)
container = ENG_CFG['adls_container']


def _latest_date_partition(prefix: str) -> str:
    """Return the lexicographically latest date-folder under prefix."""
    entries = fs.ls(f"{container}/{prefix}", detail=False)
    dates = sorted([e.split('/')[-1] for e in entries if e.split('/')[-1].isdigit()])
    if not dates:
        raise FileNotFoundError(f"No date partitions found under {prefix}")
    return dates[-1]


def read_parquet(subpath: str) -> pd.DataFrame:
    path = f"abfss://{container}@{ENG_CFG['adls_account_name']}.dfs.core.windows.net/{subpath}"
    return pd.read_parquet(path, storage_options={'account_name': ENG_CFG['adls_account_name'],
                                                   'credential': credential})


def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = f"abfss://{container}@{ENG_CFG['adls_account_name']}.dfs.core.windows.net/{subpath}"
    df.to_parquet(path, storage_options={'account_name': ENG_CFG['adls_account_name'],
                                         'credential': credential},
                  index=False, engine='pyarrow')
    logger.info('Written %d rows → %s', len(df), subpath)

In [ ]:
# Load feature matrix and labels from the latest (or configured) date partition
date_tag = ENG_CFG['feature_matrix_date'] or _latest_date_partition(ENG_CFG['feature_matrix_prefix'])
logger.info('Reading feature matrix from partition: %s', date_tag)

df = read_parquet(f"{ENG_CFG['feature_matrix_prefix']}/{date_tag}/feature_matrix.parquet")
df_labels = read_parquet(f"{ENG_CFG['labels_prefix']}/{date_tag}/labels.parquet")

country_col = ENG_CFG['country_col']
year_col    = ENG_CFG['year_col']

# Sort once; all section code assumes this ordering
df = df.sort_values([country_col, year_col]).reset_index(drop=True)

feature_cols_input = [c for c in df.columns
                      if c not in (country_col, year_col)
                      and c not in ENG_CFG['outcome_cols']]

print(f'Feature matrix : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Input features : {len(feature_cols_input)}')
print(f'Countries      : {df[country_col].nunique()}')
print(f'Years          : {df[year_col].min()}–{df[year_col].max()}')

## Section A — Variable Transformations

Adds monotone re-expressions, rate-of-change features, and trend/cycle decompositions.
None of these change the information content of a feature for tree models, but they matter
for the LASSO pass in Notebook 15 where L1 regularisation penalises raw scale.

| Transform | Columns | Rationale |
|---|---|---|
| `log1p` | GDP, population, fatalities, refugee counts | Right-skewed distributions; log linearises LASSO penalties |
| `sqrt` | Bounded event counts | Milder compression for variables that include zeros |
| First difference `_diff1` | GDP, democracy indices, FSI, CPI | Year-on-year change — captures deterioration rather than level |
| HP trend/cycle | GDP, inflation, military spend, FSI | Separates secular trend from cyclical deviation, each of which has distinct conflict relevance |


In [ ]:

# Accumulate (name, source_cols, transform) records for the catalog
_catalog: list[dict] = []

def _catalog_add(name: str, sources: list[str], transform: str) -> None:
    _catalog.append({'feature': name, 'source_cols': '|'.join(sources), 'transform': transform})


def run_transformations(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if not cfg['enable_transformations']:
        logger.info('Section A skipped (enable_transformations=False)')
        return df

    df = df.copy()
    added = []

    # ── log1p ────────────────────────────────────────────────────────────────
    for col in cfg['log_cols']:
        if col not in df.columns:
            continue
        name = f'{col}_log1p'
        df[name] = np.log1p(df[col].clip(lower=0))
        _catalog_add(name, [col], 'log1p')
        added.append(name)

    # ── sqrt ──────────────────────────────────────────────────────────────────
    for col in cfg['sqrt_cols']:
        if col not in df.columns:
            continue
        name = f'{col}_sqrt'
        df[name] = np.sqrt(df[col].clip(lower=0))
        _catalog_add(name, [col], 'sqrt')
        added.append(name)

    # ── first difference (within country) ────────────────────────────────────
    for col in cfg['diff_cols']:
        if col not in df.columns:
            continue
        name = f'{col}_diff1'
        df[name] = df.groupby(cfg['country_col'])[col].diff(1)
        _catalog_add(name, [col], 'first_difference')
        added.append(name)

    # ── Hodrick-Prescott trend / cycle ────────────────────────────────────────
    if cfg['hp_cols']:
        from statsmodels.tsa.filters.hp_filter import hpfilter

        for col in cfg['hp_cols']:
            if col not in df.columns:
                continue
            trend_name = f'{col}_hp_trend'
            cycle_name = f'{col}_hp_cycle'
            trends, cycles = [], []

            for country, grp in df.groupby(cfg['country_col']):
                series = grp[col].fillna(method='ffill').fillna(method='bfill').fillna(0)
                if len(series) < 4:
                    trends.extend([np.nan] * len(series))
                    cycles.extend([np.nan] * len(series))
                    continue
                try:
                    cycle, trend = hpfilter(series.values, lamb=cfg['hp_lambda'])
                    trends.extend(trend.tolist())
                    cycles.extend(cycle.tolist())
                except Exception:
                    trends.extend([np.nan] * len(series))
                    cycles.extend([np.nan] * len(series))

            df[trend_name] = trends
            df[cycle_name] = cycles
            _catalog_add(trend_name, [col], f'hp_trend(lambda={cfg[\"hp_lambda\"]})')
            _catalog_add(cycle_name, [col], f'hp_cycle(lambda={cfg[\"hp_lambda\"]})')
            added += [trend_name, cycle_name]

    logger.info('Section A: added %d transformed features', len(added))
    return df


df = run_transformations(df, ENG_CFG)
print(f'Columns after Section A: {df.shape[1]}  (+{df.shape[1] - len(feature_cols_input) - 2} new)')


## Section B — Interaction Effects

Creates pairwise products from `ENG_CFG['interaction_pairs']`. Each pair is
standardised (z-scored) before multiplication so the product is in
standard-deviation units and the coefficient is directly interpretable —
both by LASSO and by SHAP.

Interactions are explicitly configured rather than exhaustively enumerated.
Exhaustive enumeration over 200+ features produces ~20,000 candidates, most
of which are noise; LASSO would then need a very large regularisation path
to discard them and the MI rescue pass would rescue false positives. The
configured list encodes domain hypotheses that are worth testing:
economic stress amplified by political exclusion, food insecurity in
conflict-prone states, military over-reach in fragile settings, etc.

Add or remove pairs in `ENG_CFG['interaction_pairs']` as your theory evolves.
Each entry is `(col_a, col_b, output_name)`.


In [ ]:

def _zscore_col(series: pd.Series) -> pd.Series:
    mu, sigma = series.mean(), series.std()
    return (series - mu) / sigma if sigma > 0 else series - mu


def run_interactions(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if not cfg['enable_interactions']:
        logger.info('Section B skipped (enable_interactions=False)')
        return df

    df = df.copy()
    added = []
    skipped = []

    for col_a, col_b, out_name in cfg['interaction_pairs']:
        if col_a not in df.columns or col_b not in df.columns:
            skipped.append(out_name)
            continue
        z_a = _zscore_col(df[col_a])
        z_b = _zscore_col(df[col_b])
        df[out_name] = z_a * z_b
        _catalog_add(out_name, [col_a, col_b], 'interaction_zscore_product')
        added.append(out_name)

    if skipped:
        logger.warning('Section B: skipped %d pairs (source column absent): %s',
                       len(skipped), skipped)
    logger.info('Section B: added %d interaction features', len(added))
    return df


df = run_interactions(df, ENG_CFG)
print(f'Columns after Section B: {df.shape[1]}')


## Section C — Spatial Spillover Features

Countries are not independent units. Conflict, economic shocks, and refugee flows
diffuse across borders. These features capture that regional contagion signal by
weighting each country's value by the values of its geographic neighbors.

**Spatial weights matrix** — built once from capital city coordinates using
K-nearest neighbors (K = `spatial_k_neighbors`). The same `W` is reused for
every year, as countries do not move.

For each variable in `spatial_features` and each year:

| Feature | Column suffix | What it captures |
|---|---|---|
| Spatial lag | `_slag` | Weighted mean of neighbors' values — direct regional context |
| Local Moran's I | `_local_moran` | How similar this country's value is to its neighbors (positive = clustered, negative = outlier) |
| LISA quadrant | `_lisa_quad` | Cluster type: 1=High-High, 2=Low-High, 3=Low-Low, 4=High-Low, 0=not significant |

The spatial lag of conflict fatalities in neighbors is one of the strongest
predictors of civil war onset in the quantitative conflict literature (Gleditsch 2007).
LISA quadrant membership identifies whether a country sits in a regional hotspot
(HH) or is an isolated outlier (HL/LH).


In [ ]:

def _build_spatial_weights(geo_path: str, countries: list[str], k: int):
    """
    Build a row-standardised KNN spatial weights object from capital city
    lat/lon. Returns (W, ordered_iso3_list) where W is a libpysal weights
    object whose observations are in the same order as ordered_iso3_list.

    geo_path must point to a CSV with columns: iso3, lat, lon
    """
    import libpysal

    geo = pd.read_csv(geo_path, dtype={'iso3': str})
    geo = geo[geo['iso3'].isin(countries)].drop_duplicates('iso3').set_index('iso3')

    # Keep only countries present in both the panel and the geo file
    common = [c for c in countries if c in geo.index]
    missing_geo = set(countries) - set(common)
    if missing_geo:
        logger.warning('%d countries lack geo coordinates and will get NaN spatial features: %s',
                       len(missing_geo), sorted(missing_geo))

    coords = geo.loc[common, ['lat', 'lon']].values.tolist()
    W = libpysal.weights.KNN.from_array(coords, k=k)
    W.transform = 'r'   # row-standardise so spatial lag is a weighted mean
    return W, common


def run_spatial(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    if not cfg['enable_spatial']:
        logger.info('Section C skipped (enable_spatial=False)')
        return df

    geo_path = cfg.get('geo_coords_path')
    if not geo_path or not Path(geo_path).exists():
        logger.warning('Section C skipped — geo_coords_path not found: %s', geo_path)
        return df

    import esda

    df = df.copy()
    country_col = cfg['country_col']
    year_col    = cfg['year_col']

    all_countries = sorted(df[country_col].unique().tolist())
    W, ordered_countries = _build_spatial_weights(geo_path, all_countries, cfg['spatial_k_neighbors'])
    country_pos = {c: i for i, c in enumerate(ordered_countries)}

    years = sorted(df[year_col].unique().tolist())
    added = []

    for feat in cfg['spatial_features']:
        if feat not in df.columns:
            logger.warning('Spatial feature "%s" not in dataframe — skipping', feat)
            continue

        slag_col  = f'{feat}_slag'
        moran_col = f'{feat}_local_moran'
        quad_col  = f'{feat}_lisa_quad'

        slag_vals  = pd.Series(index=df.index, dtype=float)
        moran_vals = pd.Series(index=df.index, dtype=float)
        quad_vals  = pd.Series(index=df.index, dtype=float)

        for yr in years:
            yr_mask = df[year_col] == yr
            yr_df   = df.loc[yr_mask, [country_col, feat]].set_index(country_col)

            # Vector aligned to ordered_countries; fill missing with column median
            col_median = yr_df[feat].median()
            y = np.array([
                yr_df.loc[c, feat] if c in yr_df.index else col_median
                for c in ordered_countries
            ], dtype=float)
            # Replace remaining NaN with median
            y = np.where(np.isnan(y), np.nanmedian(y), y)

            # Spatial lag: W @ y
            slag_vec = W.sparse.dot(y)

            # Local Moran's I
            try:
                li = esda.Moran_Local(y, W, permutations=0)
                moran_vec = li.Is
                quad_vec  = li.q.astype(float)
            except Exception:
                moran_vec = np.full(len(ordered_countries), np.nan)
                quad_vec  = np.zeros(len(ordered_countries))

            # Map back to the dataframe rows for this year
            for idx in df.index[yr_mask]:
                c = df.at[idx, country_col]
                if c in country_pos:
                    pos = country_pos[c]
                    slag_vals.at[idx]  = slag_vec[pos]
                    moran_vals.at[idx] = moran_vec[pos]
                    quad_vals.at[idx]  = quad_vec[pos]

        df[slag_col]  = slag_vals
        df[moran_col] = moran_vals
        df[quad_col]  = quad_vals

        _catalog_add(slag_col,  [feat], f'spatial_lag(k={cfg["spatial_k_neighbors"]})')
        _catalog_add(moran_col, [feat], f'local_moran_I(k={cfg["spatial_k_neighbors"]})')
        _catalog_add(quad_col,  [feat], f'lisa_quadrant(k={cfg["spatial_k_neighbors"]})')
        added += [slag_col, moran_col, quad_col]

    logger.info('Section C: added %d spatial features across %d variables',
                len(added), len(cfg['spatial_features']))
    return df


df = run_spatial(df, ENG_CFG)
print(f'Columns after Section C: {df.shape[1]}')


## Section D — Feature Audit and Output

Before writing, sanity-check the derived columns: flag near-zero-variance
features (uninformative regardless of model), report missingness, and show
point-biserial correlation with each outcome label so obviously useless
features can be pruned before LASSO even sees them.


In [ ]:

# Derived column names added by this notebook
derived_cols = [entry['feature'] for entry in _catalog]

audit_rows = []
for col in derived_cols:
    if col not in df.columns:
        continue
    s = df[col]
    row = {
        'feature':     col,
        'transform':   next((e['transform'] for e in _catalog if e['feature'] == col), ''),
        'source_cols': next((e['source_cols'] for e in _catalog if e['feature'] == col), ''),
        'missing_pct': round(s.isna().mean() * 100, 1),
        'std':         round(s.std(), 4),
        'near_zero_var': s.std() < 1e-6,
    }
    # Point-biserial correlation with each outcome (where label is available)
    df_merged = df.merge(df_labels, on=[country_col, year_col], how='left')
    for outcome in ENG_CFG['outcome_cols']:
        if outcome not in df_merged.columns:
            row[f'rpb_{outcome}'] = None
            continue
        valid = df_merged[[col, outcome]].dropna()
        if valid[outcome].nunique() < 2 or len(valid) < 20:
            row[f'rpb_{outcome}'] = None
        else:
            r, _ = stats.pointbiserialr(valid[outcome], valid[col])
            row[f'rpb_{outcome}'] = round(r, 3)
    audit_rows.append(row)

audit_df = pd.DataFrame(audit_rows)

nzv = audit_df['near_zero_var'].sum()
high_missing = (audit_df['missing_pct'] > 50).sum()
print(f'Derived features     : {len(audit_df)}')
print(f'Near-zero variance   : {nzv}  (consider dropping)')
print(f'Missing > 50%        : {high_missing}')
print()
print(audit_df.to_string(index=False))


In [ ]:

# Write augmented feature matrix and feature catalog to ADLS
out_prefix = f"{ENG_CFG['output_prefix']}/{RUN_DATE}"

write_parquet(df, f'{out_prefix}/feature_matrix_engineered.parquet')
write_parquet(audit_df, f'{out_prefix}/feature_catalog_additions.csv')  # kept as parquet for consistency

# Also write the catalog as plain CSV for easy inspection
catalog_path = Path('feature_catalog_additions.csv')
audit_df.to_csv(catalog_path, index=False)
print(f'Local catalog written to: {catalog_path}')

print()
print('=' * 60)
print('Notebook 14b complete')
print('=' * 60)
print(f'  Input features    : {len(feature_cols_input)}')
print(f'  Derived features  : {len(derived_cols)}')
print(f'  Total features    : {df.shape[1] - 2}  (excl. {country_col}, {year_col})')
print(f'  Output parquet    : {out_prefix}/feature_matrix_engineered.parquet')
print()
print('Update ENG_CFG in Notebook 15 to read from:')
print(f'  processed/feature_matrix_engineered/{RUN_DATE}/feature_matrix_engineered.parquet')
